In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import ccxt
import ccxt.pro as ccxtpro
from btc_model.setting.setting import get_settings
from btc_model.core.util.crypto_util import CryptoUtil


/Users/Jason/work/source/03_ThorpAI/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


In [13]:
# 获取设置
setting = get_settings('cex.sandbox.okx')

apikey = setting['apikey']
secretkey = setting['secretkey']
passphrase = setting['passphrase']


# 初始化币安交易所
params = {
    'enableRateLimit': True,
    'proxies': {
        'http': get_settings('common')['proxies'].get('http', None),                  
        'https': get_settings('common')['proxies'].get('https', None),
    },
    'apiKey': apikey,          
    'secret': secretkey,  
    'password': passphrase,     
    'options': {
        'defaultType': 'spot',  # 可选：'spot', 'margin', 'future'
    },
    #  'headers': {
    #     'x-simulated-trading': '1'
    # }
}

exchange = ccxt.okx(params)

exchange.set_sandbox_mode(True)

exchange_pro = ccxtpro.okx(params)
exchange_pro.set_sandbox_mode(True)

Unclosed connection
client_connection: Connection<ConnectionKey(host='wspap.okx.com', port=8443, is_ssl=True, ssl=True, proxy=None, proxy_auth=None, proxy_headers_hash=None)>


In [14]:
try:
    # 获取账户余额
    balance = exchange.fetch_balance({
        'type': 'trading'
    })
    
    print("=== 模拟账户余额 ===")
    for currency in balance['total']:
        if balance['total'][currency] > 0:
            print(f"{currency}:")
            print(f"  可用: {balance['free'][currency]}")
            print(f"  冻结: {balance['used'][currency]}")
            print(f"  总额: {balance['total'][currency]}")

except Exception as e:
    print(f"错误: {str(e)}")


=== 模拟账户余额 ===
BTC:
  可用: 1.0
  冻结: 0.0
  总额: 1.0
OKB:
  可用: 100.0
  冻结: 0.0
  总额: 100.0
USDT:
  可用: 4888.426
  冻结: 0.0
  总额: 4888.426
ETH:
  可用: 1.0
  冻结: 0.0
  总额: 1.0
LSK:
  可用: 199.82
  冻结: 0.0
  总额: 199.82


In [15]:

try:
    # 现货限价单
    def place_spot_limit_order(symbol, side, amount, price):
        """
        symbol: 交易对，如 'BTC/USDT'
        side: 'buy' 或 'sell'
        amount: 数量
        price: 价格
        """
        order = exchange.create_order(
            symbol=symbol,
            type='limit',
            side=side,
            amount=amount,
            price=price
        )
        return order

    # 现货市价单
    def place_spot_market_order(symbol, side, amount):
        order = exchange.create_order(
            symbol=symbol,
            type='market',
            side=side,
            amount=amount
        )
        return order

    # 永续合约限价单
    def place_swap_limit_order(symbol, side, amount, price, leverage=1):
        """
        symbol: 交易对，如 'BTC/USDT:USDT'
        side: 'buy' 或 'sell'
        amount: 合约数量
        price: 价格
        leverage: 杠杆倍数
        """
        # 设置杠杆
        exchange.set_leverage(leverage, symbol)
        
        order = exchange.create_order(
            symbol=symbol,
            type='limit',
            side=side,
            amount=amount,
            price=price,
            params={
                'tdMode': 'cross',  # 全仓模式，'isolated' 为逐仓模式
                'posSide': 'long'   # 持仓方向，'long' 或 'short'
            }
        )
        return order


    # 示例使用
    # 现货限价买入
    min_amount = 0.01  # 假设最小交易量为 0.01
    spot_order = place_spot_limit_order(
        symbol='BTC/USDT',
        side='buy',
        amount=min_amount,  # 使用最小交易量
        price=40000         # 价格 40000 USDT
    )
    print("现货限价单:", spot_order)

    # 永续合约开多
    swap_order = place_swap_limit_order(
        symbol='BTC/USDT:USDT',
        side='buy',
        amount=min_amount,  # 使用最小交易量
        price=40000,
        leverage=2          # 2倍杠杆
    )
    print("永续合约限价单:", swap_order)

except Exception as e:
    print(f"下单错误: {str(e)}")

# 查询订单状态
def check_order_status(order_id, symbol):
    try:
        order = exchange.fetch_order(order_id, symbol)
        print(f"订单状态: {order['status']}")
        return order
    except Exception as e:
        print(f"查询订单错误: {str(e)}")
        return None

# 取消订单
def cancel_order(order_id, symbol):
    try:
        result = exchange.cancel_order(order_id, symbol)
        print("订单已取消")
        return result
    except Exception as e:
        print(f"取消订单错误: {str(e)}")
        return None

现货限价单: {'info': {'clOrdId': 'e847386590ce4dBC5592f822f91e4d32', 'ordId': '2338289305050341376', 'sCode': '0', 'sMsg': 'Order placed', 'tag': 'e847386590ce4dBC', 'ts': '1742188851704'}, 'id': '2338289305050341376', 'clientOrderId': 'e847386590ce4dBC5592f822f91e4d32', 'timestamp': None, 'datetime': None, 'lastTradeTimestamp': None, 'lastUpdateTimestamp': None, 'symbol': 'BTC/USDT', 'type': 'limit', 'timeInForce': None, 'postOnly': None, 'side': 'buy', 'price': None, 'stopLossPrice': None, 'takeProfitPrice': None, 'triggerPrice': None, 'average': None, 'cost': None, 'amount': None, 'filled': None, 'remaining': None, 'status': None, 'fee': None, 'trades': [], 'reduceOnly': False, 'fees': [], 'stopPrice': None}
永续合约限价单: {'info': {'clOrdId': 'e847386590ce4dBCa0f44fcade484080', 'ordId': '2338289318673440768', 'sCode': '0', 'sMsg': 'Order placed', 'tag': 'e847386590ce4dBC', 'ts': '1742188852110'}, 'id': '2338289318673440768', 'clientOrderId': 'e847386590ce4dBCa0f44fcade484080', 'timestamp': No

In [14]:
# 现货市价单
def place_spot_market_order(symbol, side, amount):
    order = exchange.create_order(
        symbol=symbol,
        type='market',
        side=side,
        amount=amount
    )
    return order

# 示例使用
try:
    # 使用市价单买入
    market_order = place_spot_market_order(
        symbol='BTC/USDT',
        side='buy',
        amount=0.01  # 确保数量符合最小交易量
    )
    print("现货市价单:", market_order)

except Exception as e:
    print(f"下单错误: {str(e)}")

现货市价单: {'info': {'clOrdId': 'e847386590ce4dBCbeca8fa4f69eac40', 'ordId': '2291433857851891712', 'sCode': '0', 'sMsg': 'Order placed', 'tag': 'e847386590ce4dBC', 'ts': '1740792450561'}, 'id': '2291433857851891712', 'clientOrderId': 'e847386590ce4dBCbeca8fa4f69eac40', 'timestamp': None, 'datetime': None, 'lastTradeTimestamp': None, 'lastUpdateTimestamp': None, 'symbol': 'BTC/USDT', 'type': 'market', 'timeInForce': None, 'postOnly': None, 'side': 'buy', 'price': None, 'stopLossPrice': None, 'takeProfitPrice': None, 'triggerPrice': None, 'average': None, 'cost': None, 'amount': None, 'filled': None, 'remaining': None, 'status': None, 'fee': None, 'trades': [], 'reduceOnly': False, 'fees': [], 'stopPrice': None}


In [4]:
crypto_util = CryptoUtil.get_instance()

In [9]:
crypto_util.get_funding_rate(exchange=exchange, symbol='LSK/USDT:USDT')

{}

In [ ]:
crypto_util.get_perpetual_markets(exchange=exchange)

In [10]:
markets = exchange.load_markets()

In [6]:
perpetual_markets = {}
            
for symbol, market in markets.items():
    # 筛选永续合约
    if market.get('swap') and market.get('linear'):
        market_info = {
            'symbol': symbol,
            'base': market['base'],
            'quote': market['quote'],
            'settle': market.get('settle'),
            'leverage': {
                'max': market.get('limits', {}).get('leverage', {}).get('max'),
                'min': market.get('limits', {}).get('leverage', {}).get('min', 1)
            },
            'margin_mode': market.get('margin_modes', ['isolated', 'cross']),
            'fees': {
                'maker': market.get('maker'),
                'taker': market.get('taker'),
            },
            'maintenance_margin': market.get('maintenance_margin_rate'),
            'initial_margin': market.get('initial_margin_rate'),
            'contract_size': market.get('contractSize', 1),
            'precision': {
                'price': market['precision']['price'],
                'amount': market['precision']['amount']
            }
        }
        perpetual_markets[symbol] = market_info

In [38]:
crypto_util.get_withdrawal_fees(exchange=exchange, currencies=['BTC', 'LSK'])

NameError: name 'crypto_util' is not defined

In [50]:
exchange.fetch_balance()

NetworkError: okx GET https://www.okx.com/api/v5/public/instruments?instType=SPOT

In [59]:
exchange.fetch_order_book('LSK/USDT')

RequestTimeout: okx GET https://www.okx.com/api/v5/public/instruments?instType=SPOT

In [11]:
import asyncio

await asyncio.wait_for(exchange_pro.load_markets(), 10)

{'BTC/SGD': {'id': 'BTC-SGD',
  'lowercaseId': None,
  'symbol': 'BTC/SGD',
  'base': 'BTC',
  'quote': 'SGD',
  'settle': None,
  'baseId': 'BTC',
  'quoteId': 'SGD',
  'settleId': None,
  'type': 'spot',
  'spot': True,
  'margin': False,
  'swap': False,
  'future': False,
  'option': False,
  'index': None,
  'active': True,
  'contract': False,
  'linear': None,
  'inverse': None,
  'subType': None,
  'taker': 0.0015,
  'maker': 0.001,
  'contractSize': None,
  'expiry': None,
  'expiryDatetime': None,
  'strike': None,
  'optionType': None,
  'precision': {'amount': 1e-09,
   'price': 0.1,
   'cost': None,
   'base': None,
   'quote': None},
  'limits': {'leverage': {'min': 1.0, 'max': 1.0},
   'amount': {'min': 1e-06, 'max': None},
   'price': {'min': None, 'max': None},
   'cost': {'min': None, 'max': 1000000.0}},
  'marginModes': {'cross': None, 'isolated': None},
  'created': 1734595200000,
  'info': {'alias': '',
   'auctionEndTime': '',
   'baseCcy': 'BTC',
   'category': '

In [12]:
import asyncio
symbol = 'BTC/USDT:USDT'
await asyncio.wait_for(exchange_pro.watch_ticker(symbol=symbol), 10)

{'symbol': 'BTC/USDT:USDT',
 'timestamp': 1742139260712,
 'datetime': '2025-03-16T15:34:20.712Z',
 'high': 84500.0,
 'low': 81123.2,
 'bid': 83157.0,
 'bidVolume': 748.26,
 'ask': 83159.9,
 'askVolume': 845.58,
 'vwap': None,
 'open': 84242.3,
 'close': 83159.9,
 'last': 83159.9,
 'previousClose': None,
 'change': -1082.4,
 'percentage': -1.2848652042976034,
 'average': 83701.1,
 'baseVolume': 74669414.06,
 'quoteVolume': None,
 'markPrice': None,
 'indexPrice': None,
 'info': {'instType': 'SWAP',
  'instId': 'BTC-USDT-SWAP',
  'last': '83159.9',
  'lastSz': '0.6',
  'askPx': '83159.9',
  'askSz': '845.58',
  'bidPx': '83157',
  'bidSz': '748.26',
  'open24h': '84242.3',
  'high24h': '84500',
  'low24h': '81123.2',
  'sodUtc0': '84340',
  'sodUtc8': '84327.5',
  'volCcy24h': '746694.1406',
  'vol24h': '74669414.06',
  'ts': '1742139260712'}}

In [13]:
import ccxt.pro as ccxtpro
from btc_model.setting.setting import get_settings

# 获取设置
setting = get_settings('cex.okx')


apikey = setting['apikey']
secretkey = setting['secretkey']
passphrase = setting['passphrase']


# 初始化币安交易所
params = {
    'enableRateLimit': True,
    'proxies': {
        'http': get_settings('common')['proxies']['http'],                  
        'https': get_settings('common')['proxies']['http'],
    },
    'apiKey': apikey,          
    'secret': secretkey,    
    'password': passphrase,
    'options': {
        'defaultType': 'swap',  # 可选：'spot', 'margin', 'future'
    },
    'aiohttp_proxy': get_settings('common')['proxies']['http'],
    'ws_proxy': get_settings('common')['proxies']['http']
}

# 创建交易所实例 - 注意这里是 binance 而不是 binane
exchange_pro = ccxtpro.okx(params)

In [55]:
await asyncio.wait_for(exchange_pro.load_markets(), 10)

TimeoutError: 

In [54]:
symbol = 'BTC/USDT:USDT'
await asyncio.wait_for(exchange_pro.watch_ticker(symbol=symbol), 10)

ExchangeError: okx markets not loaded

In [25]:
from btc_model.core.util.crypto_util import CryptoUtil

result = CryptoUtil.get_perpetual_markets(exchange=exchange)


{'BTC/USDT:USDT': {'symbol': 'BTC/USDT:USDT', 'base': 'BTC', 'quote': 'USDT', 'settle': 'USDT', 'leverage': {'max': 100.0, 'min': 1.0}, 'margin_mode': ['isolated', 'cross'], 'fees': {'maker': 0.0002, 'taker': 0.0005}, 'maintenance_margin': None, 'initial_margin': None, 'contract_size': 0.01, 'precision': {'price': 0.1, 'amount': 0.01}}, 'ETH/USDT:USDT': {'symbol': 'ETH/USDT:USDT', 'base': 'ETH', 'quote': 'USDT', 'settle': 'USDT', 'leverage': {'max': 100.0, 'min': 1.0}, 'margin_mode': ['isolated', 'cross'], 'fees': {'maker': 0.0002, 'taker': 0.0005}, 'maintenance_margin': None, 'initial_margin': None, 'contract_size': 0.1, 'precision': {'price': 0.01, 'amount': 0.01}}, 'SOL/USDT:USDT': {'symbol': 'SOL/USDT:USDT', 'base': 'SOL', 'quote': 'USDT', 'settle': 'USDT', 'leverage': {'max': 50.0, 'min': 1.0}, 'margin_mode': ['isolated', 'cross'], 'fees': {'maker': 0.0002, 'taker': 0.0005}, 'maintenance_margin': None, 'initial_margin': None, 'contract_size': 1.0, 'precision': {'price': 0.01, 'amo

In [32]:
import pandas as pd
pd.DataFrame(result).transpose()[['symbol', 'base', 'quote']].reset_index(drop=True)

,symbol,base,quote
0,BTC/USDT:USDT,BTC,USDT
1,ETH/USDT:USDT,ETH,USDT
2,SOL/USDT:USDT,SOL,USDT
3,TON/USDT:USDT,TON,USDT
4,DOGE/USDT:USDT,DOGE,USDT
...,...,...,...
239,ZKJ/USDT:USDT,ZKJ,USDT
240,ZRO/USDT:USDT,ZRO,USDT
241,ZRX/USDT:USDT,ZRX,USDT
242,BTC/USDC:USDC,BTC,USDC


In [18]:
order = exchange.fetch_order('2338197618470871040', 'LSK/USDT')
print(order)


{'info': {'accFillSz': '10', 'algoClOrdId': '', 'algoId': '', 'attachAlgoClOrdId': '', 'attachAlgoOrds': [], 'avgPx': '0.5637', 'cTime': '1742186119231', 'cancelSource': '', 'cancelSourceReason': '', 'category': 'normal', 'ccy': '', 'clOrdId': 'e847386590ce4dBCfbff1a8779a351ae', 'fee': '-0.008', 'feeCcy': 'LSK', 'fillPx': '0.5637', 'fillSz': '10', 'fillTime': '1742186126994', 'instId': 'LSK-USDT', 'instType': 'SPOT', 'isTpLimit': 'false', 'lever': '', 'linkedAlgoOrd': {'algoId': ''}, 'ordId': '2338197618470871040', 'ordType': 'limit', 'pnl': '0', 'posSide': 'net', 'px': '0.5637', 'pxType': '', 'pxUsd': '', 'pxVol': '', 'quickMgnType': '', 'rebate': '0', 'rebateCcy': 'USDT', 'reduceOnly': 'false', 'side': 'buy', 'slOrdPx': '', 'slTriggerPx': '', 'slTriggerPxType': '', 'source': '', 'state': 'filled', 'stpId': '', 'stpMode': 'cancel_maker', 'sz': '10', 'tag': 'e847386590ce4dBC', 'tdMode': 'cash', 'tgtCcy': '', 'tpOrdPx': '', 'tpTriggerPx': '', 'tpTriggerPxType': '', 'tradeId': '14060054'

In [ ]:
order['']


10.0